In [ ]:
# ============================================================
# FINAL FIXED Step1 Validation (no processor, direct input)
# ============================================================

import os
import torch
import torchaudio
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm

from sklearn.metrics import (
    accuracy_score, recall_score, f1_score,
    classification_report, confusion_matrix
)

from transformers import Wav2Vec2ForSequenceClassification

# -------------------------
# Paths
# -------------------------
MODEL_PATH = "/home/alpaco/kmj/stutter_step1_best"
WAV_ROOT   = "/home/alpaco/kmj/all_wavs"
VAL_CSV    = "/home/alpaco/kmj/val.csv"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("DEVICE:", device)

df = pd.read_csv(VAL_CSV)
df["binary"] = df["label6"].apply(lambda x: 0 if x == 0 else 1)

# -------------------------
# Load model only (NO PROCESSOR)
# -------------------------
model = Wav2Vec2ForSequenceClassification.from_pretrained(MODEL_PATH).to(device)
model.eval()

TARGET_SR = 16000
MAX_LEN = TARGET_SR * 3
BATCH_SIZE = 32


def fix_audio(wav, sr):
    if wav.ndim == 2:
        wav = wav.mean(dim=0)

    if sr != TARGET_SR:
        wav = torchaudio.functional.resample(wav, sr, TARGET_SR)

    if len(wav) >= MAX_LEN:
        return wav[:MAX_LEN]
    else:
        return torch.nn.functional.pad(wav, (0, MAX_LEN - len(wav)))


# ============================================================
# Batch inference (RAW input)
# ============================================================
true_labels = []
pred_labels = []

batch_wavs = []
batch_trues = []

for idx, row in tqdm(df.iterrows(), total=len(df), desc="Validating"):

    wav, sr = torchaudio.load(os.path.join(WAV_ROOT, row["filename"]))
    wav = fix_audio(wav, sr)     # shape = [48000]

    batch_wavs.append(wav)
    batch_trues.append(int(row["binary"]))

    if len(batch_wavs) == BATCH_SIZE:
        wav_tensor = torch.stack(batch_wavs).to(device)  # [B, T]

        with torch.no_grad():
            logits = model(input_values=wav_tensor).logits
            preds = logits.argmax(dim=1)

        pred_labels.extend(preds.cpu().tolist())
        true_labels.extend(batch_trues)

        batch_wavs = []
        batch_trues = []

# leftover batch
if batch_wavs:
    wav_tensor = torch.stack(batch_wavs).to(device)

    with torch.no_grad():
        logits = model(input_values=wav_tensor).logits
        preds = logits.argmax(dim=1)

    pred_labels.extend(preds.cpu().tolist())
    true_labels.extend(batch_trues)


# ============================================================
# Metrics
# ============================================================
acc = accuracy_score(true_labels, pred_labels)
macro_f1 = f1_score(true_labels, pred_labels, average="macro")
macro_recall = recall_score(true_labels, pred_labels, average="macro")

print("\n🔥 FINAL STEP1 VALIDATION RESULT 🔥")
print(f"Accuracy      : {acc:.4f}")
print(f"Macro F1      : {macro_f1:.4f}")
print(f"Macro Recall  : {macro_recall:.4f}")
print("\n--- CLASSIFICATION REPORT ---")
print(classification_report(true_labels, pred_labels, digits=4))


# ============================================================
# Confusion Matrix
# ============================================================
cm = confusion_matrix(true_labels, pred_labels)
plt.figure(figsize=(5,4))
sns.heatmap(
    cm,
    annot=True,
    fmt='d',
    cmap='Blues',
    xticklabels=["Normal", "Stutter"],
    yticklabels=["Normal", "Stutter"]
)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.title("Final Step1 Confusion Matrix")
plt.tight_layout()
plt.show()
